In [ ]:
import logging
import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["SM_FRAMEWORK"] = "tf.keras"

In [ ]:
!pip install transformers datasets seqeval torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 13.6 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16161 sha256=2d4ed86f58e5ddc83f6b5612680533dd49f14fffd71e99896057c755b4c4dd31
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import csv
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
import shutil
import torch

from collections import Counter, defaultdict
from datasets import load_dataset, ClassLabel
from seqeval.metrics import classification_report, f1_score
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch.nn.functional import cross_entropy
from torch.optim import AdamW
from torch.utils.data import DataLoader, SubsetRandomSampler
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification, AutoConfig,
    DataCollatorForTokenClassification,
    get_scheduler
)

In [ ]:
#data_path = "/content/drive/MyDrive/datasets/DNRTI/DNRTI_all_in_one.csv"
#data_path = "/content/drive/MyDrive/datasets/APTNER/aptner_clean_all_in_one.csv"
data_path = "/content/drive/MyDrive/datasets/DNRTI/DNRTI_normalized.csv"
#data_path = "/content/drive/MyDrive/datasets/APTNER_DNRTI/aptner_dnrti2.csv"
#data_path = "/content/drive/MyDrive/datasets/DNRTI/first_50_sentences.csv"
#data_path = "/content/drive/MyDrive/datasets/relations_extraction/dntri_aug_normalized.csv"
#data_path = "/content/drive/MyDrive/datasets/MS_Bulletin_Metasploit_NVD/nvd_corpus.csv"
df = pd.read_csv(data_path)
df.fillna(method = 'ffill', inplace = True)
df.head()

In [ ]:
df

In [ ]:
tags = df.tag.unique()
tags

In [ ]:
len(tags)

In [ ]:
df=df[['sentence','word','tag']]
df.rename(columns={'sentence':'Sentence', 'word': 'Word', 'tag':'Tag'},inplace=True)

In [ ]:
tag_count = df.Tag.value_counts()
tag_count

In [ ]:
df.head()

In [ ]:
GS_DATA_DIR = "/content/drive/MyDrive/datasets/DNRTI"
DATA_DIR = "/content/drive/MyDrive/datasets/DNRTI/data"

# GS_DATA_DIR = "/content/drive/MyDrive/datasets/APTNER"
# DATA_DIR  = "/content/drive/MyDrive/datasets/APTNER/data"

# GS_DATA_DIR = "/content/drive/MyDrive/datasets/relations_extraction"
# DATA_DIR = "/content/drive/MyDrive/datasets/relations_extraction/data"

# GS_DATA_DIR = "/content/drive/MyDrive/datasets/MS_Bulletin_Metasploit_NVD"
# DATA_DIR = "/content/drive/MyDrive/datasets/MS_Bulletin_Metasploit_NVD/data"

NER_FILEPATH = os.path.join(GS_DATA_DIR, "DNRTI_normalized.csv")

OUTPUT_FILEPATHS = [
  os.path.join(DATA_DIR, "train.jsonl"),
  os.path.join(DATA_DIR, "valid.jsonl"),
  os.path.join(DATA_DIR, "test.jsonl")
]

#BASE_MODEL_NAME = "bert-base-cased"
BASE_MODEL_NAME = "roberta-base"
#BASE_MODEL_NAME = "ehsanaghaei/SecureBERT"
MODEL_DIR = os.path.join(DATA_DIR, "{:s}-ner".format(BASE_MODEL_NAME))

In [ ]:
def write_output(tokens, labels, output_files, num_writter):
  assert(len(tokens) == len(labels))
  rec = json.dumps({ "tokens": tokens, "ner_tags": labels })
  dice = random.random()
  if dice <= 0.73:
    output_files[0].write("{:s}\n".format(rec))
    num_written[0] += 1
  elif dice <= 0.85:
    output_files[1].write("{:s}\n".format(rec))
    num_written[1] += 1
  else:
    output_files[2].write("{:s}\n".format(rec))
    num_written[2] += 1


os.makedirs(DATA_DIR, exist_ok=True)
output_files = [open(filepath, "w") for filepath in OUTPUT_FILEPATHS]
num_written = [0, 0, 0]
tokens, labels = [], []
with open(NER_FILEPATH, "r") as fner:
  csv_reader = csv.reader(fner)
  next(csv_reader)  # skip header
  s=2
  for row in csv_reader:
    #print(row)
    #print(int(row[0]), s, len(tokens))

    if int(row[0]) == s and len(tokens) > 0:
      # write out current sentence to train / valid / test
      write_output(tokens, labels, output_files, num_written)
      tokens, labels = [], []
      s=s+1

    elif int(row[0]) > s and len(tokens) > 0:
      # write out current sentence to train / valid / test
      write_output(tokens, labels, output_files, num_written)
      tokens, labels = [], []
      s=s+2

    # accumulate tokens and labels
    tokens.append(row[1])
    labels.append(row[2])
    # if num_written[0] > 1000:
    #   break

if len(tokens) > 0:
  write_output(tokens, labels, output_files, num_written)

[output_file.close() for output_file in output_files]
print(num_written)

In [ ]:
data_files = {
    "train": OUTPUT_FILEPATHS[0],
    "validation": OUTPUT_FILEPATHS[1],
    "test": OUTPUT_FILEPATHS[2]
}
dataset = load_dataset("json", data_files=data_files)
dataset

In [ ]:
tag_freqs_by_split = defaultdict(Counter)

for split, data in dataset.items():
    for ner_tags in data["ner_tags"]:
        prev_tag = None  # Track the previous tag to determine BIOES
        for tag in ner_tags:
            if tag.startswith("B-"):
                tag = tag.replace("B-", "")
                tag_freqs_by_split[split][tag] += 1
                prev_tag = tag
            elif tag.startswith("I-"):
                prev_tag = tag.replace("I-", "")
            elif tag.startswith("E-"):
                tag = tag.replace("E-", "")
                tag_freqs_by_split[split][tag] += 1
                prev_tag = None
            elif tag.startswith("S-"):
                tag = tag.replace("S-", "")
                tag_freqs_by_split[split][tag] += 1
                prev_tag = None
            else:
                prev_tag = None

# Convert the defaultdict to a DataFrame
pd.DataFrame.from_dict(tag_freqs_by_split, orient="index")

In [ ]:
from transformers import RobertaTokenizerFast
#BASE_MODEL_NAME = "ehsanaghaei/SecureBERT"
BASE_MODEL_NAME = "roberta-base"
tokenizer = RobertaTokenizerFast.from_pretrained(BASE_MODEL_NAME, add_prefix_space=True)
tokens = tokenizer(dataset["train"][0]["tokens"], is_split_into_words=True).tokens()
tokens

In [ ]:
input = tokenizer(dataset["train"][0]["tokens"], is_split_into_words=True)
word_ids = input.word_ids()
tokens = input.tokens()

pd.DataFrame([tokens, word_ids], index=["tokens", "word_ids"])

In [ ]:
entity_types = set()
for ner_tags in dataset["train"]["ner_tags"]:
    for ner_tag in ner_tags:
        if ner_tag.startswith("B-"):
            entity_types.add(ner_tag[2:])  # Exclude the "B-" prefix
        elif ner_tag.startswith("S-"):
            entity_types.add(ner_tag[2:])  # Exclude the "S-" prefix
entity_types = sorted(list(entity_types))
entity_types

In [ ]:
entity_types = set()
for ner_tags in dataset["train"]["ner_tags"]:
    for ner_tag in ner_tags:
        if ner_tag.startswith("B-") or ner_tag.startswith("I-") or ner_tag.startswith("S-"):
            entity_types.add(ner_tag[2:])  # Exclude prefixes to get unique entity types

entity_types = sorted(list(entity_types))

tag_names = ["O"]  # Start with "O" for outside any named entity
for entity_type in entity_types:
    tag_names.append(f"B-{entity_type}")
    tag_names.append(f"I-{entity_type}")
    # If you use "S-" (singleton) tags, add them here as well

tags = ClassLabel(names=tag_names)
label2id = {name: tags.str2int(name) + 2 for name in tag_names}
id2label = {id + 2: tags.int2str(id)  for id in range(len(tag_names))}

In [ ]:
label2id['<PAD>'] = 1
label2id['UNK'] = 0
id2label[1] = '<PAD>'
id2label[0] = 'UNK'

In [ ]:
print(label2id)
print(id2label)

In [ ]:
def tokenize_and_align_labels(tokenizer, examples, label2id, max_len):
    tokenized_inputs = tokenizer(examples["tokens"],
                                 truncation=True,
                                 padding='max_length',
                                 max_length=max_len,
                                 is_split_into_words=True)
    aligned_batch_labels = []
    for idx, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=idx)
        prev_word_id = None
        aligned_labels = []
        for word_id in word_ids:
            if word_id is None or word_id == prev_word_id:
                aligned_labels.append(0)  # Ignore tag
            else:
                label = labels[word_id] if word_id < len(labels) else 'O'  # Replace empty label with 'O'
                aligned_labels.append(label2id.get(label, 0))  # Use get method to handle missing labels
            prev_word_id = word_id
        aligned_batch_labels.append(aligned_labels)
    tokenized_inputs["labels"] = aligned_batch_labels
    return tokenized_inputs

In [ ]:
max_len = 100

train_tokenized = tokenize_and_align_labels(tokenizer, dataset["train"], label2id, max_len)
val_tokenized = tokenize_and_align_labels(tokenizer, dataset["validation"], label2id, max_len)
test_tokenized = tokenize_and_align_labels(tokenizer, dataset["test"], label2id, max_len)

In [ ]:
sample_index = 0

token_ids = train_tokenized['input_ids'][sample_index]
labels = train_tokenized['labels'][sample_index]

tokens = tokenizer.convert_ids_to_tokens(token_ids)

decoded_labels = [id2label[label] for label in labels]

for token, label in zip(tokens, decoded_labels):
    print(f'{token:15} {label}')

In [ ]:
print(train_tokenized['labels'][:1])
print(train_tokenized['input_ids'][:1])
print(train_tokenized['attention_mask'][:1])

In [ ]:
!pip install pytorch-crf

In [ ]:
import os
import torch.nn as nn
from transformers import RobertaModel, RobertaTokenizer, AutoConfig
from torchcrf import CRF

class RobertaBiGRUCRF(nn.Module):
    def __init__(self, roberta_model, num_labels, hidden_dim, dropout_rate, id2label, label2id):
        super(RobertaBiGRUCRF, self).__init__()
        self.roberta = RobertaModel.from_pretrained(roberta_model)
        self.config = self.roberta.config  # Save the config from the RoBERTa model
        self.config.id2label = id2label
        self.config.label2id = label2id
        self.dropout = nn.Dropout(dropout_rate)
        self.bigru = nn.GRU(input_size=self.roberta.config.hidden_size,
                            hidden_size=hidden_dim,
                            num_layers=1,
                            bidirectional=True,
                            batch_first=True)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)  # Output from BiGRU
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.roberta(input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        gru_output, _ = self.bigru(sequence_output)
        emissions = self.classifier(gru_output)

        if labels is not None:
            # Calculate loss using CRF
            loss = -self.crf(emissions, labels, mask=attention_mask.byte())
            return loss
        else:
            # Return the best path
            return self.crf.decode(emissions, mask=attention_mask.byte())

    def save_pretrained(self, save_directory):
        """Save the model configuration and weights to a directory."""
        if not os.path.exists(save_directory):
            os.makedirs(save_directory)

        # Save model weights
        torch.save(self.state_dict(), os.path.join(save_directory, "pytorch_model.bin"))

        # Save configuration file
        self.config.save_pretrained(save_directory)

    @classmethod
    def from_pretrained(cls, save_directory, ignore_mismatched_sizes=False):
        """Load the model configuration and weights from a directory."""
        # Load configuration
        config = AutoConfig.from_pretrained(save_directory)

        # Create model instance
        model = cls(
            roberta_model=config._name_or_path,
            num_labels=config.num_labels,
            hidden_dim=config.hidden_size,
            dropout_rate=config.hidden_dropout_prob,
            id2label=config.id2label,
            label2id=config.label2id
        )

        # Load model weights with optional ignoring of mismatched sizes
        state_dict = torch.load(os.path.join(save_directory, "pytorch_model.bin"), map_location=torch.device('cpu'))
        if ignore_mismatched_sizes:
            # Filter out mismatched keys
            state_dict = {k: v for k, v in state_dict.items() if k in model.state_dict() and model.state_dict()[k].shape == v.shape}
        model.load_state_dict(state_dict, strict=False)
        return model

In [ ]:
model = RobertaBiGRUCRF(BASE_MODEL_NAME, num_labels=len(label2id) + 1, hidden_dim=256, dropout_rate=0.1, id2label=id2label, label2id=label2id)

# Optimizer

In [ ]:
from torch.optim import AdamW
#from transformers import AdamW

def get_optimizer(model):
    # Define optimizer parameters
    no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
    roberta_params = model.roberta.named_parameters()  # Changed from bert_params to roberta_params
    additional_params = [p for n, p in model.named_parameters() if 'roberta' not in n]

    optimizer_grouped_parameters = [
        # Group 1 - Weight decay for RoBERTa params (except biases and LayerNorm)
        {'params': [p for n, p in roberta_params if not any(nd in n for nd in no_decay)],
         'weight_decay': 0.01, 'lr': 5e-5},
        # Group 2 - No weight decay for RoBERTa biases and LayerNorm
        {'params': [p for n, p in roberta_params if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0, 'lr': 5e-5},
        # Group 3 - Higher learning rate for newly added parameters (BiGRU, CRF)
        {'params': additional_params,
         'weight_decay': 0.01, 'lr': 1e-3}
    ]

    # Initialize the optimizer
    optimizer = AdamW(optimizer_grouped_parameters, lr=5e-5, eps=1e-8)
    return optimizer



In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Assuming train_tokenized and val_tokenized include input_ids, attention_mask, and labels
train_dataset = TensorDataset(torch.tensor(train_tokenized['input_ids']),
                              torch.tensor(train_tokenized['attention_mask']),
                              torch.tensor(train_tokenized['labels']))
val_dataset = TensorDataset(torch.tensor(val_tokenized['input_ids']),
                            torch.tensor(val_tokenized['attention_mask']),
                            torch.tensor(val_tokenized['labels']))

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


In [ ]:
for batch in train_loader:
    input_ids, attention_mask, labels = batch  # Unpack the tuple directly

    print("Input IDs:", input_ids)
    print("Attention Mask:", attention_mask)
    print("Labels:", labels)
    break

In [ ]:
model_filename = 'best_model_checkpoint.pt'
model_save_path = os.path.join('/content/', model_filename)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = get_optimizer(model)

num_epochs = 4

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Assuming optimizer is already defined
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)


def train_and_evaluate(model, train_loader, val_loader, optimizer, num_epochs, device):
    """Train and evaluate the model across multiple epochs focusing solely on loss."""

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    for epoch in range(num_epochs):
        model.train()  # Set the model to training mode
        total_train_loss = 0


        for input_ids, attention_mask, labels in train_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()  # Clear previous gradients
            loss = model(input_ids, attention_mask, labels)  # Compute loss directly in the model
            loss.backward()  # Backpropagate to compute gradients
            optimizer.step()  # Update model parameters

            total_train_loss += loss.item()

        # Calculate average train loss
        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        # Validation step
        model.eval()  # Set the model to evaluation mode
        total_val_loss = 0
        with torch.no_grad():
            for input_ids, attention_mask, labels in val_loader:
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                labels = labels.to(device)

                loss = model(input_ids, attention_mask, labels)
                total_val_loss += loss.item()

        # Calculate average validation loss
        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        scheduler.step(avg_val_loss)
        if avg_val_loss < best_val_loss:
          best_val_loss = avg_val_loss
          torch.save(model.state_dict(), model_save_path)
          print(f"Epoch {epoch + 1}: New optimal model saved with validation loss {best_val_loss}")

        print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")

    return train_losses, val_losses

In [ ]:
train_losses, val_losses = train_and_evaluate(model, train_loader, val_loader, optimizer, num_epochs, device)

In [ ]:
import matplotlib.pyplot as plt

# Find global minimum and maximum
min_loss = min(min(train_losses), min(val_losses))
max_loss = max(max(train_losses), max(val_losses))

# Normalize data
normalized_train_losses = [(x - min_loss) / (max_loss - min_loss) for x in train_losses]
normalized_val_losses = [(x - min_loss) / (max_loss - min_loss) for x in val_losses]

# Generate x-axis values based on the number of epochs
epochs = range(1, len(normalized_train_losses) + 1)

### Step 2: Plot the Normalized Data
plt.figure(figsize=(5, 5))
plt.plot(epochs, normalized_train_losses, label='Training Loss')
plt.plot(epochs, normalized_val_losses, label='Validation Loss')
plt.title('Training and Validation Losses')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(False)

# Setting the y-axis limits to be exactly 0 to 1 for clarity
plt.ylim([0, 1])

plt.show()

In [ ]:
test_dataset = TensorDataset(torch.tensor(test_tokenized['input_ids']),
                             torch.tensor(test_tokenized['attention_mask']),
                             torch.tensor(test_tokenized['labels']))
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
# model_path ='/content/best_model_checkpoint.pt'


# try:
#     model.load_state_dict(torch.load(model_path))
#     model.to(device)
# except Exception as e:
#     print(f"Error loading the model: {e}")

In [ ]:
model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Decode predictions from the CRF
        output = model(input_ids, attention_mask)
        predictions.extend(output)

        # Now filter out the padded labels based on attention_mask
        for label, mask in zip(labels, attention_mask):
            active_labels = torch.masked_select(label, mask.bool())
            true_labels.append(active_labels.tolist())

# Convert numerical labels back to their string representations
decoded_predictions = [[id2label[tag] for tag in sequence] for sequence in predictions]
decoded_true_labels = [[id2label[tag] for tag in sequence] for sequence in true_labels]



In [ ]:
print(decoded_predictions[:4])
print(decoded_true_labels[:4])

In [ ]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

In [ ]:
precision = precision_score(decoded_true_labels, decoded_predictions)
recall = recall_score(decoded_true_labels, decoded_predictions)
f1 = f1_score(decoded_true_labels, decoded_predictions)

# Generate the classification report
report = classification_report(decoded_true_labels, decoded_predictions)

print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)
print("\nClassification Report:\n", report)

In [ ]:
2+'p'

In [ ]:
def align_tokens_and_predicted_labels(tokens_cpu, preds_cpu):
    """
    Align predicted labels with their corresponding tokens.
    """
    aligned_tokens, aligned_preds = [], []
    id2label = model.config.id2label
    prev_token = None
    for token, pred in zip(tokens_cpu, preds_cpu):
        #print(token, prev_token)
        if not token.startswith("Ġ") and prev_token is not None:
            #prev_token += token[2:]
            prev_token += token
        else:
            if prev_token is not None:
                aligned_tokens.append(prev_token[1:])
                aligned_preds.append(id2label[prev_pred])
            prev_token = token
            prev_pred = pred
    if prev_token is not None:
        aligned_tokens.append(prev_token[1:])
       #print("Here is the previous", prev_pred)
        aligned_preds.append(id2label[prev_pred])
    return aligned_tokens, aligned_preds

def predict_labels_for_texts(texts):
    """
    Predict labels for a list of input texts.
    """
    predicted_tokens_list, predicted_tags_list = [], []
    for text in texts:
        # Ensure text is properly formatted
        text = ' ' + text + ' '

        # Tokenize and prepare inputs
        inputs = tokenizer(text, return_tensors="pt").to(device)

        # Perform inference
        with torch.no_grad():
            outputs = model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])

        # Get tokens and predictions
        tokens_cpu = tokenizer.convert_ids_to_tokens(inputs['input_ids'].view(-1))
        preds_cpu = outputs[0]  # CRF decode outputs

        # Align tokens and predictions
        aligned_tokens, aligned_preds = align_tokens_and_predicted_labels(tokens_cpu, preds_cpu)

        predicted_tokens_list.append(aligned_tokens)
        predicted_tags_list.append(aligned_preds)

    return predicted_tokens_list, predicted_tags_list


In [ ]:
texts = [
    "Marie Curie won the Nobel Prize in 1903 and 1911.",
    "Joe Biden is the current President of the United States."
]

predicted_tokens, predicted_tags = predict_labels_for_texts(texts)

print(predicted_tokens)
print(predicted_tags)


In [ ]:
df = pd.DataFrame([predicted_tokens[0], predicted_tags[0]], index=["tokens", "predicted_tags"]).T

# Display the DataFrame
df

In [ ]:
texts = [" Night Dragon was a cyber espionage campaign that targeted oil, energy, and petrochemical companies, along with individuals and executives in Kazakhstan, Taiwan, Greece, and the United States. "]
texts = [" From April 19-24 , 2017 , a politically-motivated , targeted campaign was carried out against numerous Israeli organizations "]

predicted_tokens, predicted_tags = predict_labels_for_texts(texts)

print(predicted_tokens)
print(predicted_tags)

In [ ]:
directory = "/content/drive/MyDrive/datasets/relations_extraction/dnrti_aug_re/bestModel/RoBERTaBiGRUCRF1"

# Create directory if it does not exist
if not os.path.exists(directory):
    os.makedirs(directory)

# Save vocabulary of the tokenizer
tokenizer.save_vocabulary(directory)

# Save the model weights and its configuration file
model.save_pretrained(directory)

print('All files saved')


# Load the model

In [ ]:
from transformers import RobertaTokenizer, RobertaForTokenClassification
import torch
import pandas as pd

In [ ]:
directory = "/content/drive/MyDrive/datasets/relations_extraction/dnrti_aug_re/bestModel/RoBERTaBiGRUCRF1"

In [ ]:
# Load the tokenizer
tokenizer = RobertaTokenizer.from_pretrained(directory)

# Load the model
model = RobertaBiGRUCRF.from_pretrained(directory,ignore_mismatched_sizes=True)
#model = RobertaForTokenClassification.from_pretrained(directory, ignore_mismatched_sizes=True)

# Move model to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print('Model and tokenizer loaded')

In [ ]:
id2label = model.config.id2label
id2label

In [ ]:
texts = [
    "Marie Curie won the Nobel Prize in 1903 and 1911.",
    "Joe Biden is the current President of the United States."
]

predicted_tokens, predicted_tags = predict_labels_for_texts(texts)

print(predicted_tokens)
print(predicted_tags)


# NER Inference

In [ ]:
#file_path = '/content/drive/MyDrive/datasets/aptNotes/curated/aptner_train.txt'
file_path = '/content/drive/MyDrive/datasets/aptNotes/curated/mitre_software.txt'
#file_path = '/content/drive/MyDrive/datasets/aptNotes/curated/mitre_groups.txt'
#file_path = 'updated_text'

try:
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        annotated_text = file.read()
       # print(content)
except FileNotFoundError:
    print(f"File '{file_path}' not found.")
except IOError:
    print(f"Error reading file '{file_path}'.")

print(annotated_text)

In [ ]:
# #annotated_text = annotated_text.split()
# annotated_text = annotated_text.split('\n')
# #annotated_text=annotated_text[1:-1]
# updated_text = []
# sentences = []
# sen = ''
# for word in annotated_text:
#   if word == '.':
#     e = ' '
#     sen = sen + ' ' + word
#     sentences.append(sen)
#     sen = ''
#   else:
#     sen = sen + ' ' + word

# sentences

# MITRE

In [ ]:
# sentences = [
#     "Threat Group-1314 is an unattributed threat group that has used compromised credentials to log into a victim's remote access infrastructure.",
#     'Molerats is an Arabic-speaking, politically-motivated threat group that has been operating since 2012.',
#     "The group's victims have primarily been in the Middle East, Europe, and the United States."
# ]

# # Add a space before the period at the end of each sentence
# modified_sentences = [sentence[:-1] + ' .' if sentence.endswith('.') else sentence for sentence in sentences]

# # Print the modified sentences to verify
# for sentence in modified_sentences:
#     print(sentence)


In [ ]:
import re

sent = []
#lines = annotated_text.strip().split(' . ')
lines = annotated_text.strip().split('\n\n')
for line in lines:
  #print(line)
  if line:
    line = line.replace('\n', ' ')
    sent.append(line)

#transfrom column text into row
for text in sent:
  sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text.strip())

sentences

In [ ]:
sentences = [sentence[:-1] + ' .' if sentence.endswith('.') else sentence for sentence in sentences]
sentences

In [ ]:
# #annotated_text = annotated_text.split()
# annotated_text = annotated_text.split('\n')
# #annotated_text=annotated_text[1:-1]
# updated_text = []
# sentences = []
# sen = ''
# for word in annotated_text:
#   if word == '.':
#     e = ' '
#     sen = sen + ' ' + word
#     sentences.append(sen)
#     sen = ''
#   else:
#     sen = sen + ' ' + word

# sentences

In [ ]:
predicted_tokens, predicted_tags = predict_labels_for_texts(sentences)

In [ ]:
def align_tokens_tags_to_file(predicted_tokens, predicted_tags, output_file):
    with open(output_file, 'w') as f:
        for tokens, tags in zip(predicted_tokens, predicted_tags):
            for token, tag in zip(tokens[1:-1], tags[1:-1]):  # Skip the special tokens <s> and </s>
                f.write(f"{token} {tag}\n")
            f.write("\n")  # Add a newline to separate sentences

# Example usage
# input_text = ['Marie Curie won the Nobel Prize in 1903, and 1911.', 'Joe Biden is the current President of the United States.']
# predicted_tokens = [['<s>', 'Marie', 'Curie', 'won', 'the', 'Nobel', 'Prize', 'in', '1903,', 'and', '1911.', '</s>'], ['<s>', 'Joe', 'Biden', 'is', 'the', 'current', 'President', 'of', 'the', 'United', 'States.', '</s>']]
# predicted_tags = [['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Time', 'O', 'B-Time', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']]

#output_path = '/content/drive/MyDrive/datasets/aptNotes/curated/ann_aptner.txt'
#output_path = "/content/drive/MyDrive/datasets/relations_extraction/dnrti_aug_re/curated_cti/ann_aptner1.txt"
#output_path = "/content/drive/MyDrive/datasets/relations_extraction/dnrti_aug_re/curated_cti/ann_mitre_groups1.txt"
output_path = "/content/drive/MyDrive/datasets/relations_extraction/dnrti_aug_re/curated_cti/ann_mitre_software1.txt"
align_tokens_tags_to_file(predicted_tokens, predicted_tags, output_path)